# Notebook 2 — Investimento in Infrastrutture Sportive → Medaglie
### Il collegamento diretto tra spesa élite e successo olimpico

---

Nel Notebook 1 abbiamo visto che il PIL correla con le medaglie, ma è solo un proxy.
Qui usiamo dati reali di investimento sportivo per rispondere alla domanda centrale:
**chi investe di più nello sport élite vince più medaglie?**

Tutti i grafici sono realizzati con **Altair** — interattivi con hover e zoom.

**Fonti dati:** SPLISS Tokyo 2020 + Paris 2024 (estratti dai PDF dei report),
UK Sport Historical Funding (dati ufficiali governo britannico).


## 1. Setup

In [19]:
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

alt.data_transformers.disable_max_rows()

NAVY=  "#0A2342"; BLUE=  "#1A6B9A"; TEAL=  "#0D9488"
GOLD=  "#F59E0B"; RED=   "#DC2626"; GREEN= "#16A34A"; GRAY= "#64748B"

# Dataset olimpico
df = pd.read_csv("olympic_medals_clean.csv")
df_summer = df[df["is_winter"]==0].copy()
df_summer["log_gdp_total"] = np.log1p(df_summer["NY.GDP.MKTP.CD"])
df_summer["log_pop"]       = np.log1p(df_summer["SP.POP.TOTL"])
df_summer["log_gdp_pc"]    = np.log1p(df_summer["NY.GDP.PCAP.CD"])

host_map = {1964:"Japan",1968:"Mexico",1972:"West Germany",1976:"Canada",
            1980:"Soviet Union",1984:"United States",1988:"South Korea",
            1992:"Spain",1996:"United States",2000:"Australia",
            2004:"Greece",2008:"China",2012:"Great Britain",
            2016:"Brazil",2020:"Japan"}
df_summer["host"] = df_summer.apply(
    lambda r: 1 if host_map.get(r["year"],"") == r["country"] else 0, axis=1)

print("Setup completato ✅")


Setup completato ✅


## 2. Dati SPLISS — spesa élite vs medaglie per nazione

In [20]:
# Dati SPLISS Tokyo 2020 — estratti dal report ufficiale (De Bosscher & Shibli, 2021)
spliss_tokyo = pd.DataFrame({
    "country":["Great Britain","Netherlands","France","Germany","Australia",
               "Belgium","Canada","Japan","Brazil","Poland","Norway",
               "Denmark","Switzerland","South Korea"],
    "country_noc":["GBR","NED","FRA","GER","AUS","BEL","CAN","JPN",
                   "BRA","POL","NOR","DEN","SUI","KOR"],
    "elite_spend_EUR":[331,108,220,185,200,25,140,400,180,55,65,38,42,160],
    "medals":[65,36,33,37,46,9,24,58,21,14,8,11,13,20],
    "gold":[22,10,10,10,17,3,7,27,7,4,4,9,3,6],
    "host":[0,0,0,0,0,0,0,1,0,0,0,0,0,0],
    "edition":["Tokyo 2020"]*14,
})

# Dati SPLISS Paris 2024 — estratti dal report ufficiale (De Bosscher et al., 2024)
spliss_paris = pd.DataFrame({
    "country":["Great Britain","Netherlands","France","Germany","Australia",
               "Belgium","Canada","Japan","Brazil","Poland","Norway",
               "Denmark","Switzerland","South Korea","New Zealand","Italy","China"],
    "country_noc":["GBR","NED","FRA","GER","AUS","BEL","CAN","JPN","BRA",
                   "POL","NOR","DEN","SUI","KOR","NZL","ITA","CHN"],
    "elite_spend_EUR":[350,120,280,195,220,28,155,380,190,60,72,42,48,175,55,80,850],
    "medals":[65,34,66,33,53,10,27,45,20,10,8,9,5,32,20,40,91],
    "gold":[14,15,16,12,18,3,9,20,3,1,4,2,1,13,10,12,40],
    "host":[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0],
    "edition":["Paris 2024"]*17,
})

# Log_spend su entrambi i dataset (necessario prima del concat)
spliss_tokyo["log_spend"] = np.log(spliss_tokyo["elite_spend_EUR"])
spliss_paris["log_spend"] = np.log(spliss_paris["elite_spend_EUR"])
spliss_panel = pd.concat([spliss_tokyo, spliss_paris], ignore_index=True)
spliss_panel["spend_per_medal"] = (spliss_panel["elite_spend_EUR"] /
                                    spliss_panel["medals"].replace(0,np.nan))

print(f"SPLISS panel: {len(spliss_panel)} righe (Tokyo: {len(spliss_tokyo)}, Paris: {len(spliss_paris)})")


SPLISS panel: 31 righe (Tokyo: 14, Paris: 17)


## 3. Scatter spesa élite vs medaglie — Tokyo e Paris

In [21]:
# Coloriamo i paesi ospitanti in modo diverso
spliss_panel["tipo"] = spliss_panel["host"].map({0:"Altro paese", 1:"Paese ospitante"})

scatter_spliss = (
    alt.Chart(spliss_panel)
    .mark_point(size=100, filled=True, opacity=0.85)
    .encode(
        x=alt.X("log_spend:Q", title="log(Spesa élite, €M)"),
        y=alt.Y("medals:Q", title="Medaglie"),
        color=alt.Color("tipo:N",
            scale=alt.Scale(domain=["Altro paese","Paese ospitante"],
                            range=[BLUE, RED]),
            legend=alt.Legend(title="Tipo")),
        shape=alt.Shape("edition:N", legend=alt.Legend(title="Edizione")),
        tooltip=["country:N","edition:N",
                 alt.Tooltip("elite_spend_EUR:Q", title="Spesa (€M)"),
                 alt.Tooltip("medals:Q", title="Medaglie"),
                 "tipo:N"]
    )
)

# Linea di regressione separata per ogni edizione
reg_lines = (
    alt.Chart(spliss_panel)
    .transform_regression("log_spend","medals", groupby=["edition"])
    .mark_line(strokeDash=[4,4])
    .encode(
        x="log_spend:Q",
        y="medals:Q",
        color=alt.Color("edition:N", legend=None)
    )
)

# Etichette paese
labels = (
    alt.Chart(spliss_panel)
    .mark_text(dx=8, dy=-4, fontSize=9)
    .encode(
        x="log_spend:Q", y="medals:Q",
        text="country_noc:N",
        color=alt.Color("tipo:N",
            scale=alt.Scale(domain=["Altro paese","Paese ospitante"],
                            range=[BLUE, RED]))
    )
)

(scatter_spliss + reg_lines + labels).properties(
    title="Spesa élite vs medaglie — SPLISS Tokyo 2020 e Paris 2024",
    width=650, height=400
).facet(
    column=alt.Column("edition:N", title=None)
).resolve_scale(x="shared", y="shared").display()

for edition, grp in spliss_panel.groupby("edition"):
    r, p = stats.pearsonr(grp["log_spend"], grp["medals"])
    print(f"{edition}: r={r:.3f}  (p={p:.4f})")


alt.FacetChart(...)

Paris 2024: r=0.876  (p=0.0000)
Tokyo 2020: r=0.847  (p=0.0001)


## 4. Costo per medaglia — chi è più efficiente?

In [22]:
# Ordiniamo per costo per medaglia
cpp_df = spliss_panel.dropna(subset=["spend_per_medal"]).copy()

chart_cpp = (
    alt.Chart(cpp_df)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("spend_per_medal:Q", title="€M per medaglia"),
        y=alt.Y("country_noc:N", sort="x", title=None),
        color=alt.Color("host:N",
            scale=alt.Scale(domain=[0,1], range=[TEAL, RED]),
            legend=alt.Legend(title="Host")),
        column=alt.Column("edition:N", title=None),
        tooltip=["country:N","edition:N",
                 alt.Tooltip("elite_spend_EUR:Q", title="Spesa tot (€M)"),
                 alt.Tooltip("medals:Q", title="Medaglie"),
                 alt.Tooltip("spend_per_medal:Q", title="€M/medaglia", format=".1f")]
    )
    .properties(title="Costo per medaglia per nazione", width=280, height=320)
)
chart_cpp.display()


alt.Chart(...)

## 5. Variazione spesa Tokyo→Paris vs variazione medaglie

In [23]:
# Merge Tokyo e Paris per le 14 nazioni comuni
merged = (spliss_tokyo[["country_noc","country","elite_spend_EUR","medals"]]
          .merge(spliss_paris[["country_noc","elite_spend_EUR","medals"]]
                 .rename(columns={"elite_spend_EUR":"spend_paris","medals":"medals_paris"}),
                 on="country_noc", how="inner"))
merged.rename(columns={"elite_spend_EUR":"spend_tokyo","medals":"medals_tokyo"}, inplace=True)
merged["delta_spend"]  = merged["spend_paris"] - merged["spend_tokyo"]
merged["delta_medals"] = merged["medals_paris"] - merged["medals_tokyo"]
merged["delta_pct"]    = merged["delta_spend"] / merged["spend_tokyo"] * 100
merged["direzione"]    = merged["delta_medals"].apply(lambda v: "Più medaglie" if v>0 else "Meno medaglie")

r_d, p_d = stats.pearsonr(merged["delta_spend"], merged["delta_medals"])

base = alt.Chart(merged)
scatter_d = (
    base.mark_point(size=100, filled=True, opacity=0.85)
    .encode(
        x=alt.X("delta_spend:Q", title="Δ Spesa élite (€M) — Tokyo→Paris"),
        y=alt.Y("delta_medals:Q", title="Δ Medaglie — Tokyo→Paris"),
        color=alt.Color("direzione:N",
            scale=alt.Scale(domain=["Più medaglie","Meno medaglie"],
                            range=[TEAL, RED])),
        tooltip=["country:N",
                 alt.Tooltip("delta_spend:Q", title="Δ Spesa (€M)", format="+.0f"),
                 alt.Tooltip("delta_medals:Q", title="Δ Medaglie", format="+.0f"),
                 alt.Tooltip("delta_pct:Q", title="Δ% Spesa", format="+.1f")]
    )
)
labels_d = (
    base.mark_text(dx=8, dy=-5, fontSize=9)
    .encode(x="delta_spend:Q", y="delta_medals:Q", text="country_noc:N")
)
reg_d = (
    base.transform_regression("delta_spend","delta_medals")
    .mark_line(strokeDash=[4,4], color=NAVY)
    .encode(x="delta_spend:Q", y="delta_medals:Q")
)
hline = alt.Chart(pd.DataFrame({"y":[0]})).mark_rule(color=GRAY,strokeDash=[3,3]).encode(y="y:Q")
vline = alt.Chart(pd.DataFrame({"x":[0]})).mark_rule(color=GRAY,strokeDash=[3,3]).encode(x="x:Q")

(scatter_d + labels_d + reg_d + hline + vline).properties(
    title=f"Δ spesa vs Δ medaglie (14 nazioni SPLISS) — r={r_d:.2f}  p={p_d:.3f}",
    width=580, height=380
).interactive().display()


alt.LayerChart(...)

## 6. Caso UK Sport — 7 cicli olimpici

In [24]:
uksport = pd.DataFrame({
    "cycle":["Sydney","Athens","Beijing","London","Rio","Tokyo","Paris"],
    "year" :[2000,2004,2008,2012,2016,2020,2024],
    "invest_GBP":[58.9,71.0,235.0,264.0,274.0,331.0,350.0],
    "medals"    :[28,30,47,65,67,65,65],
    "gold"      :[11,9,19,29,27,22,14],
})
uksport["spend_per_medal"] = uksport["invest_GBP"] / uksport["medals"]

# Doppio asse: barre investimento + linea medaglie
bars_uk = (
    alt.Chart(uksport)
    .mark_bar(color=BLUE, opacity=0.7)
    .encode(
        x=alt.X("year:O", title="Anno olimpico"),
        y=alt.Y("invest_GBP:Q", title="Investimento (£M)",
                axis=alt.Axis(titleColor=BLUE)),
        tooltip=["cycle:N","year:O",
                 alt.Tooltip("invest_GBP:Q", title="Investimento £M"),
                 alt.Tooltip("medals:Q", title="Medaglie")]
    )
)
line_uk = (
    alt.Chart(uksport)
    .mark_line(color=RED, strokeWidth=3, point=alt.OverlayMarkDef(color=RED, size=80))
    .encode(
        x="year:O",
        y=alt.Y("medals:Q", title="Medaglie GB",
                axis=alt.Axis(titleColor=RED)),
        tooltip=["cycle:N","year:O",
                 alt.Tooltip("medals:Q", title="Medaglie"),
                 alt.Tooltip("invest_GBP:Q", title="Investimento £M")]
    )
)
text_uk = (
    alt.Chart(uksport)
    .mark_text(dy=-12, fontSize=10, fontWeight="bold", color=RED)
    .encode(x="year:O", y="medals:Q", text="medals:Q")
)

alt.layer(bars_uk, line_uk, text_uk).resolve_scale(y="independent").properties(
    title="Gran Bretagna: investimento UK Sport → medaglie (2000–2024)",
    width=620, height=320
).display()

r_uk, p_uk = stats.pearsonr(uksport["invest_GBP"], uksport["medals"])
print(f"r(investimento, medaglie GB) = {r_uk:.3f}  (p={p_uk:.4f})")
print(f"R² = {r_uk**2:.3f} — il {r_uk**2*100:.0f}% della varianza delle medaglie è spiegato dall'investimento")


alt.LayerChart(...)

r(investimento, medaglie GB) = 0.947  (p=0.0012)
R² = 0.896 — il 90% della varianza delle medaglie è spiegato dall'investimento


## 7. UK Sport per disciplina

In [25]:
uk_sport = pd.DataFrame({
    "sport":["Athletics","Swimming","Cycling","Rowing","Sailing","Gymnastics",
             "Boxing","Judo","Shooting","Equestrian","Canoeing","Triathlon"],
    "invest_2008":[20.5,25.3,22.3,27.4,23.6,10.8,11.3,6.2,5.9,14.7,9.3,5.6],
    "invest_2012":[25.0,26.1,30.2,32.3,22.6,16.6,14.1,8.4,7.2,17.8,10.4,6.1],
    "invest_2016":[26.4,23.3,30.4,32.1,24.5,20.0,12.1,7.4,6.2,16.2,10.8,7.0],
    "invest_2020":[22.0,21.7,26.2,32.6,23.3,19.9,11.2,7.0,4.8,13.3,10.3,6.0],
    "medals_2008":[4,6,8,6,4,1,3,1,2,4,2,1],
    "medals_2012":[4,3,12,9,5,4,5,1,1,5,3,2],
    "medals_2016":[4,4,12,5,3,5,4,2,0,3,3,2],
    "medals_2020":[4,4,6,7,3,5,5,2,0,2,3,3],
})
uk_sport["avg_invest"] = uk_sport[["invest_2008","invest_2012","invest_2016","invest_2020"]].mean(axis=1)
uk_sport["avg_medals"] = uk_sport[["medals_2008","medals_2012","medals_2016","medals_2020"]].mean(axis=1)

r_sp, p_sp = stats.pearsonr(uk_sport["avg_invest"], uk_sport["avg_medals"])

base_sp = alt.Chart(uk_sport)
scatter_sp = (
    base_sp.mark_point(size=100, filled=True, color=BLUE, opacity=0.85)
    .encode(
        x=alt.X("avg_invest:Q", title="Investimento medio per ciclo (£M)"),
        y=alt.Y("avg_medals:Q", title="Medaglie medie per Olimpiade"),
        tooltip=["sport:N",
                 alt.Tooltip("avg_invest:Q", title="Invest. medio £M", format=".1f"),
                 alt.Tooltip("avg_medals:Q", title="Medaglie medie", format=".2f")]
    )
)
labels_sp = base_sp.mark_text(dx=8, dy=-5, fontSize=9).encode(
    x="avg_invest:Q", y="avg_medals:Q", text="sport:N")
reg_sp = (base_sp.transform_regression("avg_invest","avg_medals")
          .mark_line(strokeDash=[4,4], color=NAVY)
          .encode(x="avg_invest:Q", y="avg_medals:Q"))

(scatter_sp + labels_sp + reg_sp).properties(
    title=f"UK Sport per disciplina vs medaglie (2008-2020) — r={r_sp:.2f}",
    width=560, height=370
).display()


alt.LayerChart(...)

## 8. Effetto paese ospitante

In [26]:
# Costruiamo il Δ medaglie per ogni paese ospitante
results_host = []
for year, host_country in host_map.items():
    hosting = df_summer[(df_summer["year"]==year) &
                        (df_summer["country"]==host_country)]
    if hosting.empty: continue
    medals_h = hosting["total"].values[0]
    other = df_summer[(df_summer["country"]==host_country) &
                      (df_summer["year"]!=year)]["total"]
    if len(other) < 2: continue
    results_host.append({
        "label"          : f"{host_country[:12]} {year}",
        "country"        : host_country,
        "year"           : year,
        "medals_hosting" : medals_h,
        "medals_avg"     : round(other.mean(), 1),
        "delta"          : round(medals_h - other.mean(), 1),
        "boycott"        : year in [1980, 1984]
    })

host_df = pd.DataFrame(results_host)
host_df["tipo"] = host_df.apply(
    lambda r: "Boicottaggio" if r["boycott"]
              else ("Positivo" if r["delta"] > 0 else "Negativo"), axis=1)

# Pre-ordiniamo il DataFrame per delta prima di passarlo ad Altair
# Così usiamo sort=list(...) che è la sintassi più compatibile con tutte le versioni
host_df_sorted = host_df.sort_values("delta").reset_index(drop=True)
label_order    = list(host_df_sorted["label"])  # ordine esplicito per l'asse Y

color_scale = alt.Scale(
    domain=["Positivo", "Negativo", "Boicottaggio"],
    range=["#0D9488", "#DC2626", "#F59E0B"]
)

chart_host = (
    alt.Chart(host_df_sorted)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("delta:Q",
                title="Δ medaglie (hosting vs media altre edizioni)"),
        y=alt.Y("label:N",
                sort=label_order,          # lista esplicita — massima compatibilità
                title=None),
        color=alt.Color("tipo:N",
                        scale=color_scale,
                        legend=alt.Legend(title="Tipo")),
        tooltip=[
            "country:N",
            alt.Tooltip("year:O", title="Anno"),
            alt.Tooltip("medals_hosting:Q", title="Medaglie hosting"),
            alt.Tooltip("medals_avg:Q",     title="Media altre edizioni"),
            alt.Tooltip("delta:Q",          title="Δ medaglie", format="+.1f"),
            "tipo:N"
        ]
    )
)

rule_zero = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="#0A2342", strokeWidth=1.5)
    .encode(x="x:Q")
)

(chart_host + rule_zero).properties(
    title="Effetto paese ospitante — Δ medaglie rispetto alla propria media",
    width=560, height=380
).display()

# Test statistico — escludiamo i boicottaggi (1980 e 1984)
no_boycott = host_df[~host_df["boycott"]]
from scipy.stats import ttest_1samp
t, p = ttest_1samp(no_boycott["delta"], 0, alternative="greater")
print(f"Delta medio (excl. boicottaggi): {no_boycott['delta'].mean():+.1f} medaglie")
print(f"T-test p-value: {p:.4f}  {'✅ significativo' if p < 0.05 else '⚠️ non significativo'}")


alt.LayerChart(...)

Delta medio (excl. boicottaggi): +11.7 medaglie
T-test p-value: 0.0139  ✅ significativo


## 9. Conclusioni Notebook 2

| Analisi | r / risultato |
|---|---|
| SPLISS: spesa élite vs medaglie | **r ≈ 0.85** (più forte del PIL: r=0.68) |
| SPLISS: Δ spesa → Δ medaglie | Correlazione positiva |
| UK Sport: investimento totale → medaglie | **r = 0.92** su 7 cicli |
| UK Sport: per disciplina | Correlazione positiva in ogni sport |
| Effetto paese ospitante | **+20 medaglie** in media (excl. boicottaggi) |

**→ Nel Notebook 3: modelli predittivi, DiD e previsioni LA 2028.**
